<a href="https://colab.research.google.com/github/Hmnth7/Python-lab-experiments/blob/main/EXPERIMENT_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BESCOM Smart Electricity Consumption Monitoring System Using Generators and Iterators

## Problem Statement
A utility company receives continuous electricity readings. Develop an Electricity Consumption Monitoring System Using Generators and Iterators. This system should:

*   Process data sequentially.
*   Not load the complete dataset into memory.
*   Use generators for continuous readings.
*   Implement iterator behavior.
*   Identify abnormal or high consumption readings.

---

### Implementation Steps: `ElectricityReading` Class and `generate_readings()` Generator

In [ ]:
import time
import random

class ElectricityReading:
    """Represents a single electricity consumption reading."""
    def __init__(self, timestamp, consumption):
        self.timestamp = timestamp
        self.consumption = consumption

    def __str__(self):
        return f"[Timestamp: {self.timestamp}, Consumption: {self.consumption:.2f} kWh]"

    def __repr__(self):
        return self.__str__()


def generate_readings(num_readings=10, max_consumption=100, anomaly_chance=0.1):
    """Generates a sequence of electricity readings, occasionally introducing anomalies."""
    print(f"Generating {num_readings} electricity readings...")
    current_time = time.time()
    for i in range(num_readings):
        timestamp = current_time + i * 60 # Readings every minute

        # Introduce an occasional anomaly (very high consumption)
        if random.random() < anomaly_chance:
            consumption = random.uniform(max_consumption * 1.5, max_consumption * 3) # Anomaly
            print(f"  --> ANOMALY GENERATED at {time.strftime('%H:%M:%S', time.localtime(timestamp))}: {consumption:.2f} kWh")
        else:
            consumption = random.uniform(10, max_consumption) # Normal consumption

        yield ElectricityReading(timestamp, consumption)
        time.sleep(0.1) # Simulate some delay in reading generation

    print("Finished generating readings.")

### Implementation Steps: `ConsumptionMonitor` Iterator Class

In [ ]:
class ConsumptionMonitor:
    """Monitors electricity consumption readings, acting as an iterator.
    It calculates total/average consumption and identifies high readings.
    """
    def __init__(self, reading_generator, threshold=150):
        self.reading_generator = reading_generator
        self.threshold = threshold
        self.total_consumption = 0
        self.reading_count = 0
        self.high_consumption_readings = []

    def __iter__(self):
        return self

    def __next__(self):
        try:
            reading = next(self.reading_generator)
            self.reading_count += 1
            self.total_consumption += reading.consumption

            if reading.consumption > self.threshold:
                self.high_consumption_readings.append(reading)
                print(f"  [ALERT] High consumption detected: {reading.consumption:.2f} kWh at {time.strftime('%H:%M:%S', time.localtime(reading.timestamp))}")

            return reading
        except StopIteration:
            print("\n--- Monitoring Complete ---")
            print(f"Total readings processed: {self.reading_count}")
            if self.reading_count > 0:
                print(f"Average consumption: {self.total_consumption / self.reading_count:.2f} kWh")
            if self.high_consumption_readings:
                print(f"High consumption readings identified ({len(self.high_consumption_readings)}):")
                for r in self.high_consumption_readings:
                    print(f"  - {r}")
            else:
                print("No high consumption readings detected above threshold.")
            raise # Re-raise StopIteration to end the iteration

### Implementation Steps: Execute and Verify Output

In [ ]:
# 1. Create a reading generator
# Let's generate 20 readings with a slightly higher chance of anomaly for demonstration
readings_gen = generate_readings(num_readings=20, max_consumption=100, anomaly_chance=0.2)

# 2. Create a ConsumptionMonitor instance
# Set a threshold for high consumption, e.g., 120 kWh
monitor = ConsumptionMonitor(readings_gen, threshold=120)

# 3. Process readings using the iterator
print("\nStarting electricity consumption monitoring...")
for reading in monitor:
    # We can perform real-time actions with each reading here if needed
    # For this example, the monitor handles printing alerts and statistics
    pass

print("\nMonitoring process finished.")


Starting electricity consumption monitoring...
Generating 20 electricity readings...
  --> ANOMALY GENERATED at 02:57:23: 209.97 kWh
  [ALERT] High consumption detected: 209.97 kWh at 02:57:23
  --> ANOMALY GENERATED at 03:03:23: 240.00 kWh
  [ALERT] High consumption detected: 240.00 kWh at 03:03:23
Finished generating readings.

--- Monitoring Complete ---
Total readings processed: 20
Average consumption: 76.11 kWh
High consumption readings identified (2):
  - [Timestamp: 1788404243.9843142, Consumption: 209.97 kWh]
  - [Timestamp: 1788404603.9843142, Consumption: 240.00 kWh]

Monitoring process finished.
